In [1]:
import torch
from torch import nn
import pandas as pd
import random
import optuna
import numpy as np
from torch.optim.lr_scheduler import OneCycleLR

In [2]:
df = pd.read_csv("wo_bos_eos/no_sos_eos_attention_gru_reversal_data.csv")
                     
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)
print(len(df))

cuda
162584


In [3]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
print(device)

cuda


In [4]:
# tree_type = "right"
tree_val = 10
# if tree_type == "left":
#     df['source'] = df['source'].apply(lambda x: ([tree_val] + eval(x)))
# elif tree_type == "right":
#     df['source'] = df['source'].apply(lambda x: ([tree_val] + eval(x)))

df['source'] = df['source'].apply(lambda x: ([10] + eval(x)))
df['output'] = df['output'].apply(lambda x: eval(x))

In [5]:
seed_num = 1
random.seed(seed_num)
np.random.seed(seed_num)
torch.manual_seed(seed_num)
torch.cuda.manual_seed_all(seed_num)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [6]:
df_sorted = df.sort_values(by='source', key=lambda col: col.map(len))

In [7]:
model_size = len(df_sorted['output'].iloc[0])
print(model_size)   
batch_size = 128

128


In [8]:
left_role = 0
right_role = 1
def right_recurse(lst):
    batch_size = lst[0].shape[0]
    tensors = [[x, torch.full((batch_size,), left_role, device = device)] for x in lst[:-1]]
    
    acc = lst[-1]
    for t in reversed(tensors):
        acc = [t, [acc, torch.full((batch_size,), right_role, device = device)]]
    return acc

def left_recurse(lst):
    batch_size = lst[0].shape[0]
    tensors = [[x, torch.full((batch_size,), right_role, device = device)] for x in lst[1:]]
    
    acc = lst[0]
    for t in tensors:
        acc = [[acc, torch.full((batch_size,), left_role, device = device)], t]
    return acc

In [9]:
def collate_lists(*lists):
    return [torch.tensor([lst[i] for lst in lists]) for i in range(len(lists[0]))]

In [10]:
sources = df_sorted["source"].tolist()
targets = df_sorted["output"].tolist()

from collections import defaultdict

buckets = defaultdict(list) 

for src, trg in zip(sources, targets):
    buckets[len(src)].append((src, trg))

del buckets[8]
del buckets[9]

print(buckets.keys())

dict_keys([2, 3, 4, 5, 6, 7])


In [11]:
src_batches = []
trg_batches = []

for seq_len in sorted(buckets.keys()):
    pairs = buckets[seq_len]
    for i in range(0, len(pairs), batch_size):
        # If i + batch_size is out of bounds, it just goes to the end of the list
        chunk = pairs[i:i + batch_size]
        src_batch = [p[0] for p in chunk]
        trg_batch = [p[1] for p in chunk]

        collated = collate_lists(*src_batch) 
        # nested = right_recurse(collated)
        nested = left_recurse(collated)

        src_batches.append(nested)
        trg_batches.append(torch.stack([torch.tensor(v, dtype=torch.float32) for v in trg_batch]))

n = len(src_batches)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

combined = list(zip(src_batches, trg_batches))
random.shuffle(combined)
src_batches, trg_batches = zip(*combined)

train_src, train_trg = src_batches[:n_train], trg_batches[:n_train]
val_src, val_trg = src_batches[n_train:n_train+n_val], trg_batches[n_train:n_train+n_val]
test_src, test_trg = src_batches[n_train+n_val:], trg_batches[n_train+n_val:]

In [12]:
def get_recursive_batch_size(x):
    if isinstance(x[0], list):
        return get_recursive_batch_size(x[0])
    return x[0].shape[0]

class RTPREncode(nn.Module):
    def __init__(self, nfillers, nroles, filler_size, role_size, model_size):
        super(RTPREncode, self).__init__()
        self.fillers = nn.Embedding(nfillers, filler_size)
        self.roles = nn.Embedding(nroles, role_size)
        self.filler_size = filler_size
        self.role_size = role_size
        self.model_size = model_size
        # self.dropout = nn.Dropout(0.4)
        self.w = torch.nn.Parameter(torch.randn(filler_size, filler_size*role_size)*0.01)
        self.wo = torch.nn.Parameter(torch.randn(model_size, filler_size)*0.01)

    def forward(self, x):
        cur_batch_size = get_recursive_batch_size(x)

        result = torch.zeros(cur_batch_size, self.filler_size, self.role_size, device=self.w.device)

        for pair in x:
            filler = pair[0]
            role = pair[1].to(device)

            if isinstance(filler, list):
                filler = self(filler)
            else:
                # if filler[0].item() == tree_val:
                #     filler = torch.zeros(cur_batch_size, self.filler_size, device=device)
                # else:
                filler = self.fillers(filler.to(device))

            role = self.roles(role)
            result += torch.einsum("bf,br->bfr", filler, role)

        result = result.view(cur_batch_size, -1)
        result = torch.einsum("ij,bj->bi", self.w, result)
        return result

In [13]:
#Evaluate a model (for validation and testing loss)
def evaluate(model, src, trg, criterion, device):
    with torch.no_grad():
        model.eval()
        epoch_loss = 0.0

        for i in range(len(src)):
            src_batch = src[i]
            trg_batch = trg[i].to(device)
            encoding = model(src_batch)
            output = torch.matmul(encoding, model.wo.T)    # (batch_size, model_size)
            loss = criterion(output, trg_batch)
            epoch_loss += loss.item()

        return epoch_loss / len(src)

In [14]:
def objective(trial):
    random.seed(seed_num)
    np.random.seed(seed_num)
    torch.manual_seed(seed_num)
    torch.cuda.manual_seed_all(seed_num)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    num_epochs = 50

    model = RTPREncode(12, 2, 64, 64, model_size).to(device)
    learning_rate = trial.suggest_float("lr", 1e-5, 1e-3, log = True)
    norm_val = trial.suggest_float("grad_norm", 1, 10, log = True)
    max_rate = trial.suggest_float("scheduler_max", 1e-4, 1e-2, log = True)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    criterion = nn.MSELoss()
    scheduler = OneCycleLR(optimizer, max_lr=max_rate, steps_per_epoch=len(train_src), epochs=num_epochs)

    for epoch in range(num_epochs):
        # print(f"Epoch {epoch+1} / {num_epochs}")
        # print(f"Trial {trial.number}")
        model.train()

        epoch_loss = 0.0
        with torch.set_grad_enabled(True):
            for i in range(len(train_src)):
                src_batch = train_src[i]
                trg_batch = train_trg[i].to(device)
                optimizer.zero_grad()

                encoding = model(src_batch)
                output = torch.matmul(encoding, model.wo.T)

                loss = criterion(output, trg_batch)
                loss.backward()

                #Norm=magnitude, clipping gradient to ensure they don't explode
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = norm_val)
                optimizer.step()
                scheduler.step()
                epoch_loss += loss.item()

        # print(f"Train Loss: {epoch_loss/len(train_src)}")
        val_loss = evaluate(model, val_src, val_trg, criterion, device)

        trial.report(val_loss, epoch)
        #prune cuts off trial early
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        # print(f"Validation Loss: {val_loss}")

    return val_loss
    

In [15]:
from optuna.samplers import TPESampler

sampler = TPESampler(seed=seed_num)

study = optuna.create_study(sampler=sampler, direction='minimize')
study.optimize(objective, n_trials=8)
print("Best Hyperparameters:", study.best_params) 

[I 2026-07-31 04:28:32,663] A new study created in memory with name: no-name-b749ba3b-a0b2-440a-b347-681cbb72b7fa
[I 2026-07-31 04:30:17,556] Trial 0 finished with value: 9.296970279137895e-05 and parameters: {'lr': 6.82407842709883e-05, 'grad_norm': 5.251997291427971, 'scheduler_max': 0.00010005268542378313}. Best is trial 0 with value: 9.296970279137895e-05.
[I 2026-07-31 04:31:58,859] Trial 1 finished with value: 0.00010767676254049134 and parameters: {'lr': 4.024066464714605e-05, 'grad_norm': 1.40202542955541, 'scheduler_max': 0.0001529949829431265}. Best is trial 0 with value: 9.296970279137895e-05.
[I 2026-07-31 04:33:39,960] Trial 2 finished with value: 0.00012304033572780264 and parameters: {'lr': 2.3578730726467393e-05, 'grad_norm': 2.215953929680391, 'scheduler_max': 0.0006216342697722397}. Best is trial 0 with value: 9.296970279137895e-05.
[I 2026-07-31 04:35:21,110] Trial 3 finished with value: 0.00010415688873101978 and parameters: {'lr': 0.0001195730942971637, 'grad_norm'

Best Hyperparameters: {'lr': 2.56392017331631e-05, 'grad_norm': 7.552964377119545, 'scheduler_max': 0.00011344234265835253}


In [17]:
pruned_trials = study.get_trials(deepcopy=False, states=[optuna.trial.TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[optuna.trial.TrialState.COMPLETE])

print("Study statistics: ")
print("  Number of finished trials: ", len(study.trials))
print("  Number of pruned trials: ", len(pruned_trials))
print("  Number of complete trials: ", len(complete_trials))

print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

Study statistics: 
  Number of finished trials:  8
  Number of pruned trials:  3
  Number of complete trials:  5
Best trial:
  Value:  8.857554572924445e-05
  Params: 
    lr: 2.56392017331631e-05
    grad_norm: 7.552964377119545
    scheduler_max: 0.00011344234265835253


In [18]:
# Train with optimized hyperparams
random.seed(seed_num)
np.random.seed(seed_num)
torch.manual_seed(seed_num)
torch.cuda.manual_seed_all(seed_num)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

num_epochs = 50

model = RTPREncode(12, 2, 64, 64, model_size).to(device)
lr = 2.56392017331631e-05
grad_norm = 7.552964377119545
scheduler_max = 0.00011344234265835253

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
criterion = nn.MSELoss()
scheduler = OneCycleLR(optimizer, max_lr=scheduler_max, steps_per_epoch=len(train_src), epochs=num_epochs)

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1} / {num_epochs}")
    # print(f"Trial {trial.number}")
    model.train()

    epoch_loss = 0.0
    with torch.set_grad_enabled(True):
        for i in range(len(train_src)):
            src_batch = train_src[i]
            trg_batch = train_trg[i].to(device)
            optimizer.zero_grad()

            encoding = model(src_batch)
            output = torch.matmul(encoding, model.wo.T)

            loss = criterion(output, trg_batch)
            loss.backward()

            #Norm=magnitude, clipping gradient to ensure they don't explode
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = grad_norm)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

    print(f"Train Loss: {epoch_loss/len(train_src)}")
    val_loss = evaluate(model, val_src, val_trg, criterion, device)
    print(f"Validation Loss: {val_loss}")

Epoch 1 / 50
Train Loss: 0.13088891381350354
Validation Loss: 0.058481762101027096
Epoch 2 / 50
Train Loss: 0.049881178263057734
Validation Loss: 0.040182591881603
Epoch 3 / 50
Train Loss: 0.025889155584210715
Validation Loss: 0.012630124840970066
Epoch 4 / 50
Train Loss: 0.004997199205694873
Validation Loss: 0.0011100039006454278
Epoch 5 / 50
Train Loss: 0.0013377590997386314
Validation Loss: 0.0006494103247611995
Epoch 6 / 50
Train Loss: 0.0005727560588798621
Validation Loss: 0.0004066110029518975
Epoch 7 / 50
Train Loss: 0.0007123427111076695
Validation Loss: 0.00035498258991786065
Epoch 8 / 50
Train Loss: 0.0008946681974178527
Validation Loss: 0.0002764978805564831
Epoch 9 / 50
Train Loss: 0.0006264367719154829
Validation Loss: 0.0008075757782948626
Epoch 10 / 50
Train Loss: 0.0007681037305023854
Validation Loss: 0.00023217237040543378
Epoch 11 / 50
Train Loss: 0.0006159650856870914
Validation Loss: 0.00037473175540402406
Epoch 12 / 50
Train Loss: 0.0005232525902421252
Validation L

In [19]:
print(evaluate(model, test_src, test_trg, criterion, device))

5.2217559539385953e-05


In [20]:
torch.save(model.state_dict(), "no_sos_bos_attention_left_reversal_2.pt")